# Import Libraries

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Read Dataset

In [2]:
flight_data = pd.DataFrame({
    "days_before_flight": 
[26,64,81,82,56,14,77,34,72,83,25,49,33,46,5,35,13,2,75,38,85,82,25,18,68,71,44,65,80,24,28,71,11,58,28,45,15,8,56,80,65,41,28,6,54,72,51,60,2,4,9,88,82,69,73],
    "distance_km": 
[575,2245,1234,3510,2458,1752,564,1229,685,1291,4245,718,319,3456,2142,743,793,1140,1362,2259,419,4258,4171,3104,3216,3504,2533,2669,2088,4486,930,4032,662,3818,4016,3190,4378,1985,1231,2813,2866,4244,2077,614,3949,1424,4340,1105,1929,2600,3550,4009,2305,2051,2358],
    "num_stops": 
[0,1,0,1,0,0,2,2,2,0,0,2,0,1,1,1,0,1,2,2,2,2,0,1,2,2,0,0,2,1,1,2,0,1,1,2,2,1,2,2,1,1,0,2,1,2,1,0,1,0,2,0,2,2,1],
    "ticket_price_usd": 
[335,380,275,391,458,431,61,331,133,233,683,214,308,506,476,298,402,480,162,390,63,425,705,569,368,400,474,375,233,703,354,443,380,492,571,475,663,495,207,287,401,595,453,357,544,189,552,254,476,609,611,456,224,246,315]
})

flight_data.head()

,days_before_flight,distance_km,num_stops,ticket_price_usd
0,26,575,0,335
1,64,2245,1,380
2,81,1234,0,275
3,82,3510,1,391
4,56,2458,0,458


# Split Dataset

In [4]:
attrs = ['days_before_flight', 'distance_km', 'num_stops']
target = 'ticket_price_usd'
X_train, X_test, y_train, y_test = train_test_split(flight_data[attrs], flight_data[target], random_state=42, test_size=0.2)

# Create Pipeline and Train

In [8]:
def create_knn_pipeline(n_neighbors):
    return make_pipeline(
            StandardScaler(),
            KNeighborsRegressor(n_neighbors=n_neighbors))

knn_pipeline = create_knn_pipeline(5)
knn_pipeline

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kneighborsregressor', KNeighborsRegressor())])

In [9]:
knn_pipeline.fit(X_train, y_train)
y_preds = knn_pipeline.predict(X_test)

In [11]:
metrics_dict = {'mae': [mean_absolute_error(y_test, y_preds)],
               'mse': [mean_squared_error(y_test, y_preds)],
               'r2': [r2_score(y_test, y_preds)]}

metrics_df = pd.DataFrame(metrics_dict)
metrics_df

,mae,mse,r2
0,34.309091,2498.650909,0.794706


In [12]:
r2_scores = []
neighbors = [1, 3, 5, 7, 9, 15]
for k in neighbors:
    knn_pipeline = create_knn_pipeline(k)
    knn_pipeline.fit(X_train, y_train)
    y_pred = knn_pipeline.predict(X_test)
    r2_scores.append(r2_score(y_test, y_pred))

In [13]:
r2_df = pd.DataFrame({'k': neighbors, 'r2': r2_scores})
r2_df

,k,r2
0,1,0.705613
1,3,0.769163
2,5,0.794706
3,7,0.762208
4,9,0.854028
5,15,0.730370


In [15]:
knn = create_knn_pipeline(9)
knn.fit(X_train, y_train)
test_df = pd.DataFrame({'days_before_flight': [20], 'distance_km': [2000], 'num_stops': [1]})
knn.predict(test_df)

array([436.88888889])

# Reason

الف: رابطه صعودی یا نزولی نیست و نوسان دارد. مقدار بهینه همسایگی 9 است.

ب: همسایگی یک تنها یک همسایه را در نظر میگیرد و چنانچه نزدیکترین همسایه‌اش نویز باشد آن را یاد میگیرد. اگر همه داده‌ها را در نظر بگیریم، آن وقت پیش‌بینی برابر با میانگین همه خواهد شد.

ج: چون مقیاسها متفاوت خواهند بود. برای مثال اهمیت تغییر کیلومتر به اندازه 1 واحد متفاوت از تغییر روز به اندازه 1 واحد خواهد بود. برای محاسبات ما تغییر یک واحدی روزها مهمتر خواهد بود ولی چنانچه استاندارد نکنیم همه هم ارزش  خواهند شد. کیلومتر معمولا در مقیاس 10 یا 1000 تفاوت خواهد کرد و مدل بیشتر به سمت کمتر کردن فاصله در بعد کیلومتر-مکان خواهد رفت.

د- مدل آموزش نمی‌بیند و پارامتر ندارد. آموزش صرفا شامل ذخیره داده‌های آموزش در حافظه مدل است. در  فرایند آزموش یا پیش بینی فاصله هر داده با تمامی داده‌های آموزش محاسبه خواهد شد. در این حالت آموزش سریع ولی آزمون کند خواهد شد. هر چه تعداد داده‌ها بیشتر شود در آزمون یا پیش بینی محاسبات بیشتر خواهد شد و هزینه بیشتری خواهیم پرداخت. 